# Распознавание цифр MNIST: KNN и SVM

Сравниваем два классических классификатора на одинаковых данных:
KNN с двумя соседями и SVM с RBF-ядром. Изображения представлены
векторами пикселей без обучения дополнительных признаков.

**Цели:** понять различия между ближайшими соседями и методом опорных
векторов, оценить test accuracy и посмотреть отдельные предсказания.

Выполняйте ячейки сверху вниз. Нужны NumPy, Matplotlib, scikit-learn
и TensorFlow для загрузки MNIST. Первый запуск может скачивать данные.
Обучение SVM и предсказания KNN на полном MNIST могут занимать заметное время.

## 1. Данные и условия сравнения

Используем стандартное разделение: 60 000 обучающих и 10 000 тестовых
изображений размером 28 × 28. Пиксели переводим в `float32` и делим
на 255, затем разворачиваем изображение в вектор из 784 признаков.

Оба классификатора получают одинаковые train и test. Параметры моделей
фиксированы; подбор по test не выполняется.
Seed фиксирует выбор примеров для иллюстрации. При этих настройках
KNN и SVM не используют случайное обучение.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

seed = 42
rng = np.random.default_rng(seed)

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.astype("float32")
x_train /= 255.0
x_test = x_test.astype("float32")
x_test /= 255.0
X_train = x_train.reshape(len(x_train), -1)
X_test = x_test.reshape(len(x_test), -1)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

## 2. KNN: ближайшие соседи

KNN присваивает объекту класс ближайших обучающих примеров.
Используем `n_neighbors=2`, евклидово расстояние и одинаковый вес соседей.
Малое число соседей делает решение чувствительным к локальным особенностям
данных. Здесь каждый пиксель рассматривается как отдельный признак.

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=2)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)
knn_accuracy = accuracy_score(y_test, y_pred_knn)
print(f"Test accuracy KNN: {knn_accuracy:.4f}")

## 3. SVM: метод опорных векторов

Используем RBF-ядро, `C=1.0`, `gamma="scale"`. Ядро позволяет строить нелинейную
границу между классами. `C` задаёт силу штрафа за ошибки, а `gamma`
влияет на масштаб локального действия RBF-ядра.

In [ ]:
svm_model = SVC(kernel="rbf", C=1.0, gamma="scale")
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
svm_accuracy = accuracy_score(y_test, y_pred_svm)
print(f"Test accuracy SVM: {svm_accuracy:.4f}")

## 4. Оценка и примеры предсказаний

Accuracy — доля правильных предсказаний среди 10 000 тестовых объектов.
Общая метрика не показывает, какие цифры чаще путаются между собой.
Несколько случайных примеров служат иллюстрацией, а не отдельной оценкой качества.

Для рисунка используем уже вычисленные предсказания: повторно вызывать
классификаторы для каждого изображения не требуется.

In [ ]:
print(f"{'Модель':<8} {'Train':>8} {'Test accuracy':>15}")
for name, accuracy in (("KNN", knn_accuracy), ("SVM", svm_accuracy)):
    print(f"{name:<8} {len(X_train):>8} {accuracy:>15.4f}")

num_images = 5
indices = rng.choice(len(X_test), size=num_images, replace=False)
fig, axes = plt.subplots(1, num_images, figsize=(15, 4))
for ax, idx in zip(axes, indices):
    ax.imshow(x_test[idx], cmap="gray", vmin=0, vmax=1)
    ax.set_title(
        f"Цифра: {y_test[idx]}\nKNN: {y_pred_knn[idx]}\nSVM: {y_pred_svm[idx]}"
    )
    ax.axis("off")
fig.tight_layout()
plt.show()

## 5. Интерпретация и ограничения

Историческая test accuracy: **0.9627 для KNN** и **0.9792 для SVM**.
Метрики текущей версии ещё не получены.

Условия сравнения по данным одинаковые, но один запуск без подбора
параметров не позволяет сделать общий вывод о превосходстве метода.
Здесь также не измеряется время обучения и предсказания.
